In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [58]:
df = pd.read_parquet(r"C:\Users\91891\Downloads\customer-churn-intelligence-system\data\optimized_df.parquet")
df

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,36,Male,UK,Manchester,1.6,15,42.700001,10.3,37.700001,1,...,42.799999,7,3,NaN,23.299999,3,1067.469971,3028,0,Q4
1,45,Female,Canada,Vancouver,2.9,10,24.799999,1.6,70.900002,1,...,0.000000,4,1,NaN,8.800000,<NA>,1289.750000,2317,0,Q4
2,56,Female,USA,New York,2.6,10,38.400002,14.8,41.700001,9,...,41.400002,2,5,85.900002,31.000000,3,2340.919922,2674,0,Q1
3,35,Male,Japan,Tokyo,3.5,6,21.900000,6.9,74.400002,0,...,16.000000,8,2,14.300000,11.200000,2,1995.430054,2418,1,Q1
4,56,Male,Germany,Berlin,1.9,24,46.400002,13.9,36.200001,5,...,35.500000,4,6,68.800003,42.900002,5,3003.570068,2657,0,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,33,Male,France,Nice,4.5,6,NaN,6.9,67.099998,2,...,7.100000,8,3,3.200000,12.900000,5,917.830017,1298,1,Q4
39996,18,Female,France,Lyon,4.8,8,21.000000,8.4,69.500000,3,...,17.400000,6,5,15.400000,14.600000,2,813.109985,3565,1,Q2
39997,38,Female,USA,Los Angeles,10.0,10,35.799999,8.8,51.200001,3,...,9.200000,5,1,20.700001,25.200001,1,2365.979980,3499,0,Q2
39998,37,Male,USA,Chicago,1.4,21,32.400002,9.6,32.500000,2,...,17.600000,5,3,38.599998,12.100000,4,1293.599976,3154,0,Q3


### So before splitting data
* remove duplicates
* invalid values
* basic cleaning
##### If we are trying to impute missing values, or do encoding or any sort of technique which involves the full data values, then that is data leakage, so fit on train and trasform on train and test.

In [47]:
print(df.duplicated().sum())
# no duplicate rows
print(df.columns.duplicated().sum())
# any duplicate column, which checks same column names
duplicate_cols = []
for i in range(len(df.columns)):
    for j in range(i+1,len(df.columns)):
        if df.iloc[:,i].equals(df.iloc[:,j]):
            #  here .iloc[rows,cols], all rows, and in the column position
            # equals() is a pandas method used to check whether, 2 pandas objects are exactly equal or not.
            # It compares both series and dataframes.
            # It returns True if all values are euqal else false.
            duplicate_cols.append(df.columns[i],df.columns[j])
print(duplicate_cols)
# which checks the values, if the column names are different, but values are same,

0
0
[]


### Problems to be addressed
1. Certain records violate expected business relationships where Average_Order_Value exceeds Lifetime_Value
but the percentage is very low, which is 0.29% ~ 0.3%.
Given the very low occurrence rate, these records were retained while documenting the limitation as a potential synthetic data artifact.

2. Age col as min is 5 annd max is 200, both are to be addressed.
3. Cart_Abandonment_Rate is the percentage it exceeding 100, as max value is 143.

4. Total_purchases min value is -13.

5. discount_rate also exceeded 100.


### AGE

In [48]:
len(df[df["Age"] < 18])

26

In [49]:
len(df[df["Age"] > 90])

16

### Age validation
* The Age column contained unrealistic values below 18 and above 90.
* Since the dataset represents ecommerce customer behavior, ages outside the 18–90 range were considered invalid for the business context.
* A total of 42 invalid records (~0.08% of the dataset) were removed during preprocessing.

In [50]:
# df = df[(df["Age"] < 18) | (df["Age"] > 90)]
# if im doing the above operation, it is also removing the rows where age is NaN, so include that also
# df = df[(df["Age"].isna()) | ((df["Age"] >= 18) & (df["Age"] <= 90))]

In [51]:
df

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
570,5,Female,USA,Los Angeles,0.6,12,34.500000,9.0,36.400002,3,...,8.600000,6,5,25.299999,16.500000,1,1236.790039,<NA>,1,Q2
1412,5,Female,USA,Los Angeles,10.0,4,24.799999,4.9,71.099998,<NA>,...,0.000000,6,1,6.400000,16.000000,1,612.940002,799,1,Q1
2103,10,Male,USA,Los Angeles,2.0,5,24.200001,NaN,75.900002,3,...,3.200000,8,2,0.000000,13.900000,4,1182.219971,3824,1,Q1
5860,10,Female,Germany,Hamburg,0.6,1,NaN,7.0,79.099998,0,...,0.000000,8,1,29.000000,9.700000,2,981.940002,0,1,Q2
6265,5,Male,France,Lyon,1.9,10,20.799999,7.5,60.200001,2,...,18.700001,6,2,0.000000,NaN,1,1698.670044,1802,1,Q1
7044,200,Female,USA,New York,2.1,9,27.600000,3.1,57.500000,1,...,3.500000,6,1,36.000000,11.300000,4,662.869995,1170,0,Q2
7500,5,Female,UK,Glasgow,3.4,22,47.700001,12.3,37.799999,11,...,28.400000,2,6,50.200001,29.700001,4,2651.139893,4335,1,Q4
8769,10,Female,USA,Chicago,2.3,5,30.700001,7.8,45.799999,2,...,19.200001,11,1,NaN,9.400000,3,985.880005,1076,1,Q2
9200,10,Female,Canada,Vancouver,2.8,8,26.000000,5.2,52.700001,3,...,0.000000,8,1,34.900002,12.900000,3,1103.010010,2117,0,Q2
9569,5,Other,USA,Los Angeles,2.5,8,NaN,7.2,93.400002,1,...,14.600000,8,0,0.000000,12.600000,2,1082.520020,1518,1,Q1


In [36]:
# After validating the remaining feature values in affected records, invalid values were replaced with NaN and will be handled later during missing value imputation.

42

In [55]:
df.loc[(df["Age"] < 18) | (df["Age"] > 90),"Age"] = np.nan

In [56]:
df

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,NaN,Male,UK,Manchester,1.6,15,42.700001,10.3,37.700001,1,...,42.799999,7,3,NaN,23.299999,3,1067.469971,3028,0,Q4
1,NaN,Female,Canada,Vancouver,2.9,10,24.799999,1.6,70.900002,1,...,0.000000,4,1,NaN,8.800000,<NA>,1289.750000,2317,0,Q4
2,NaN,Female,USA,New York,2.6,10,38.400002,14.8,41.700001,9,...,41.400002,2,5,85.900002,31.000000,3,2340.919922,2674,0,Q1
3,NaN,Male,Japan,Tokyo,3.5,6,21.900000,6.9,74.400002,0,...,16.000000,8,2,14.300000,11.200000,2,1995.430054,2418,1,Q1
4,NaN,Male,Germany,Berlin,1.9,24,46.400002,13.9,36.200001,5,...,35.500000,4,6,68.800003,42.900002,5,3003.570068,2657,0,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,NaN,Male,France,Nice,4.5,6,NaN,6.9,67.099998,2,...,7.100000,8,3,3.200000,12.900000,5,917.830017,1298,1,Q4
39996,NaN,Female,France,Lyon,4.8,8,21.000000,8.4,69.500000,3,...,17.400000,6,5,15.400000,14.600000,2,813.109985,3565,1,Q2
39997,NaN,Female,USA,Los Angeles,10.0,10,35.799999,8.8,51.200001,3,...,9.200000,5,1,20.700001,25.200001,1,2365.979980,3499,0,Q2
39998,NaN,Male,USA,Chicago,1.4,21,32.400002,9.6,32.500000,2,...,17.600000,5,3,38.599998,12.100000,4,1293.599976,3154,0,Q3


### Percentage exceeding 100
* Features to validate
* Cart Abandonment Rate
* Discount Usage Rate
* So these should not exceed 100 and not be less than 0, but as we saw in univariate analysis(max and min) value.
* So convert these values to NaN

In [66]:
len(df[df["Cart_Abandonment_Rate"] < 0])

0

In [67]:
len(df[df["Cart_Abandonment_Rate"] > 100])

25

In [69]:
df.loc[df["Cart_Abandonment_Rate"] > 100,"Cart_Abandonment_Rate"]= np.nan

In [70]:
len(df[df["Discount_Usage_Rate"] < 0])

0

In [71]:
len(df[df["Discount_Usage_Rate"] > 100])

165

In [72]:
df.loc[df["Discount_Usage_Rate"] > 100,"Discount_Usage_Rate"] = np.nan

In [78]:
df[df["Total_Purchases"] < 0][["Total_Purchases","Returns_Rate"]]
# as in total_purchases, sometimes, negative also can mean returns,
# for example they bought 10 items, in a time window, but data is calculated in another window where they returned
# 13 items, that means before that window puchased items, then it would be -3, but if we check
# that with return rate that is normal only, so decided to replace with NaN


# The Total_Purchases feature contained negative values, which is inconsistent with expected ecommerce purchasing behavior.
# Initially, the possibility of refund or return-based business logic was considered. However, corresponding Returns_Rate values remained within normal ranges and did not support unusually high return activity.
# Since both features would typically be expected to represent behavior within a similar observation period, the negative purchase values were treated as unrealistic or synthetic data artifacts.
# Therefore, invalid purchase values were replaced with NaN and will later be handled during missing value treatment.

,Total_Purchases,Returns_Rate
2319,-9.1,9.8
3294,-6.0,4.3
3487,-10.4,8.9
4046,-11.7,8.7
6268,-3.0,8.2
8128,-2.6,3.9
9151,-11.7,7.9
9312,-10.0,5.5
9924,-5.2,1.1
10487,-7.0,1.8


In [79]:
df.loc[df["Total_Purchases"] < 0,"Total_Purchases"] = np.nan